# Plantilla de documentación: CSV Preanalisis de Prueba (CPP)

## Introducción

CSV Preanalisis de Prueba (CPP) será una aplicación web sencilla para ayudar a estudiantes a hacer una primera inspección de archivos CSV antes de comenzar un análisis de datos o un trabajo de aprendizaje automático.

El usuario podrá cargar un archivo y consultar un informe con el número de filas y columnas, los tipos de datos detectados y los valores vacíos encontrados por columna y por fila. La aplicación también podrá pedir a un modelo de IA sugerencias de preguntas que se podrían investigar y posibles formas de tratar los valores ausentes.

El análisis del archivo se calculará con Python en el servidor. Para proteger los datos y reducir el uso del modelo, a OpenRouter solo se enviará un resumen estadístico: no se enviarán las filas ni los valores originales del CSV. Las recomendaciones serán orientativas y la aplicación no modificará automáticamente los datos.

## Propósito de este documento

Este notebook es la documentación de diseño del proyecto. Recoge la idea, los requisitos, los diagramas UML, el plan de trabajo, la solución técnica, las pruebas y la guía de ejecución. Cuando una parte ya está implementada se indica expresamente; el resto describe el diseño objetivo que se irá construyendo fase a fase.

## Índice general

1. Descripción detallada del proyecto
   1.1. Funcionamiento general
   1.2. Ejemplo práctico de uso
2. Objetivos de aprendizaje
3. Requisitos y stack tecnológico
   3.1. Requisitos del entorno
   3.2. Stack tecnológico
4. Estructura del proyecto
5. Requisitos funcionales y no funcionales
   5.1. Requisitos funcionales
   5.2. Requisitos no funcionales
6. Diagramas UML del sistema
   6.1. Diagrama de casos de uso
   6.2. Diagrama de clases
   6.3. Diagrama de actividad
   6.4. Diagrama de secuencia
7. Plan de ejecución paso a paso
8. Solución técnica
   8.1. Configuración del proyecto
   8.2. Backend y análisis de CSV
   8.3. Integración con OpenRouter
   8.4. Interfaz de usuario
   8.5. Tipos y contrato de datos
9. Pruebas automatizadas
   9.1. Pruebas del backend
   9.2. Pruebas del frontend
10. Guía de ejecución y despliegue
    10.1. Ejecución local
    10.2. Ejecución de pruebas
    10.3. Variables de entorno
    10.4. Despliegue

## Índice de diagramas

| Nº | Diagrama | Sección |
|----|----------|---------|
| D1 | Casos de uso | 6.1 |
| D2 | Clases | 6.2 |
| D3 | Actividad | 6.3 |
| D4 | Secuencia | 6.4 |

## Índice de requisitos

Los requisitos funcionales (RF-01 a RF-08) y no funcionales (RNF-01 a RNF-08) están numerados en la sección 5. Cada requisito se referencia por su código en el resto del documento.


## 1. Descripción detallada del proyecto

CSV Preanalisis de Prueba (CPP) será una aplicación web para revisar rápidamente un archivo CSV antes de empezar un análisis para una asignatura. La idea es que los estudiantes podamos entender cómo está organizado el conjunto de datos y detectar algunos problemas habituales, sin tener que escribir código para cada comprobación.

La aplicación mostrará cuántas filas y columnas tiene el archivo, qué tipo de dato parece haber en cada columna y dónde hay valores vacíos. También podrá pedir a una IA propuestas de preguntas para investigar y algunas opciones para tratar los valores ausentes. La IA será una ayuda para pensar, no decidirá por el estudiante ni cambiará el archivo.

### 1.1. Funcionamiento general

1. El usuario elige un archivo CSV desde la página web.
2. La interfaz, construida con Vue, envía el archivo al backend de Python.
3. El backend comprueba que el archivo se pueda leer y utiliza pandas para calcular sus dimensiones, los tipos detectados y los valores vacíos por columna y por fila.
4. El backend crea un resumen estadístico. El archivo y sus filas no se envían a OpenRouter: el modelo recibe solo ese resumen.
5. OpenRouter devuelve sugerencias de preguntas de análisis y posibles formas de tratar los valores ausentes. Son ideas iniciales que el estudiante debe valorar según el contexto de su trabajo.
6. La página muestra el informe y las sugerencias. Si la IA no está disponible, el informe calculado por Python se sigue mostrando.

### 1.2. Ejemplo práctico de uso

Una estudiante carga el CSV de una práctica con 500 filas y 6 columnas. CPP indica el tamaño del archivo, detecta cuáles columnas parecen numéricas o categóricas y señala que el 4 % de los valores de edad están vacíos. Después, la IA propone preguntas que la estudiante podría investigar y opciones para estudiar esos valores ausentes, como comprobar por qué faltan o comparar distintas formas de tratarlos. La aplicación no modifica el CSV ni asegura que una opción sea siempre la correcta.

## 2. Objetivos de aprendizaje

- Practicar cómo se construye una página web con Vue y cómo se muestran estados como carga, resultado y error.
- Aprender a recibir un archivo CSV y analizarlo con Python y pandas.
- Definir y validar los datos que entran y salen del servidor con Pydantic, y describirlos también con tipos de TypeScript en la parte web.
- Probar una conexión con un modelo de IA mediante OpenRouter, enviándole solo un resumen estadístico y manteniendo la clave fuera del navegador.
- Añadir pruebas básicas y aprender a publicar una aplicación con el frontend y el backend en proveedores distintos: la interfaz en Vercel y el servidor de análisis dockerizado en AWS (ver sección 10.4 para el porqué de este reparto).

## 3. Requisitos y stack tecnológico

Esta es una propuesta inicial para el proyecto. Las versiones exactas y las decisiones que todavía dependen de la asignatura se podrán concretar más adelante.

### 3.1. Requisitos del entorno

- **Visual Studio Code:** editor para trabajar en el proyecto y abrir este notebook.
- **Python 3.12:** lenguaje usado en el servidor y para analizar los CSV.
- **Node.js:** necesario para instalar y ejecutar las herramientas de Vue.
- **Git:** para guardar el historial de cambios y conectar el proyecto con un repositorio.
- **Clave de OpenRouter:** solo se necesitará para probar la conexión real con el modelo. Se guardará en el servidor, no en el navegador; las pruebas podrán usar respuestas simuladas.
- **Cuenta de Vercel:** necesaria para publicar la interfaz (frontend). Vercel alojará únicamente la parte estática/SPA, no el backend de Python (ver la corrección en 3.2 y el detalle en 10.4).
- **Cuenta de AWS:** necesaria para publicar el backend (FastAPI) como contenedor Docker. En la fase de pruebas convivirá con el frontend en Vercel; más adelante está previsto mover también el frontend a AWS para tener todo el proyecto en un único proveedor.

### 3.2. Stack tecnológico

#### Interfaz web

- **Vue 3:** herramienta para construir la página que permite elegir el CSV y muestra el informe y las sugerencias.
- **TypeScript:** añade tipos a JavaScript; ayuda a detectar errores y a mantener coherentes los datos que se intercambian entre la página y el servidor.
- **Tailwind CSS:** conjunto de clases de estilo que permite dar formato a la interfaz directamente desde los componentes. Se puede usar si se mantiene el enfoque del ejemplo de clase.
- **Fetch API:** función que ya ofrece el navegador para enviar el archivo al servidor y recibir el informe; no hace falta instalar una librería aparte.

#### Servidor y análisis de datos

- **Python:** se encargará de recibir el archivo, analizarlo y preparar el informe.
- **FastAPI:** librería para crear las rutas del servidor, por ejemplo, la ruta que recibe el CSV y devuelve el análisis.
- **pandas:** librería para leer archivos CSV y calcular filas, columnas, tipos y valores ausentes.
- **python-multipart:** dependencia que FastAPI necesita para recibir archivos enviados desde un formulario web.
- **Pydantic:** permite comprobar que los datos y las respuestas tengan la forma esperada antes de procesarlos o mostrarlos.
- **OpenAI Python SDK:** librería que se puede configurar para comunicarse con OpenRouter, ya que este ofrece una API compatible. La clave y la dirección del servicio se configuran en el servidor.
- **Uvicorn:** servidor ASGI que arranca FastAPI tanto en desarrollo local como dentro del contenedor Docker en producción (en AWS).

#### Pruebas y publicación

- **Pytest y HTTPX:** herramientas para probar el servidor y sus rutas sin tener que abrir la página web.
- **Vitest, Vue Test Utils y jsdom:** herramientas para probar componentes Vue como si estuvieran en una página, pero sin abrir un navegador real.
- **Vercel:** plataforma elegida para publicar únicamente la interfaz web (el build estático de Vite).
- **AWS (App Runner o ECS/Fargate):** plataforma elegida para publicar el backend como contenedor Docker, ejecutando FastAPI/Uvicorn como un proceso real.

> **Corrección:** en una versión anterior de este documento se proponía usar Vercel también para "ejecutar las funciones Python del servidor". Esa idea se descarta: Vercel está pensado para funciones serverless de vida corta o sitios estáticos, no para un servidor ASGI persistente ni para contenedores Docker a medida, y pandas junto al resto de dependencias de Python superaría con facilidad los límites de tamaño de sus funciones serverless. Por eso el backend se despliega aparte, en AWS. El detalle completo de este razonamiento está en la sección 10.4.

## 4. Estructura del proyecto

La estructura separa la interfaz (`frontend/`) del servidor de análisis (`backend/`). El estado de cada carpeta se indica entre paréntesis.

```text
csv-proyecto01-desarrollo/
├── README.md                         (existe)
├── notebooks/
│   └── plantilla_documentacion.ipynb (este documento)
├── frontend/                         (existe, Fase 1)
│   ├── src/
│   │   ├── App.vue
│   │   ├── main.js
│   │   ├── components/
│   │   │   ├── AppHeader.vue         (maqueta)
│   │   │   ├── CsvDropzone.vue       (maqueta, sin lógica)
│   │   │   ├── SummaryPanel.vue      (maqueta con datos de relleno)
│   │   │   └── SuggestionsPanel.vue  (maqueta, hueco para la IA)
│   │   └── styles/
│   │       └── main.css              (estilos base y modo oscuro)
│   ├── package.json
│   └── vite.config.js
└── backend/                          (previsto, Fase 2)
    ├── app/
    │   ├── main.py                   (rutas FastAPI)
    │   ├── analisis.py               (lectura y cálculo con pandas)
    │   ├── ia.py                     (cliente de OpenRouter)
    │   └── schemas.py                (modelos Pydantic)
    ├── tests/
    │   ├── test_analisis.py
    │   └── test_rutas.py
    ├── Dockerfile
    └── requirements.txt
```

> **Nota (corregida):** inicialmente se planteó mover `backend/` a una carpeta `api/` para encajar con las funciones Python de Vercel. Esa idea se descarta: el backend sigue siendo un servicio FastAPI independiente, empaquetado con Docker y desplegado en AWS, no una función serverless de Vercel. El frontend es la única parte que se publica en Vercel. El porqué de esta decisión se explica en las secciones 3.2 y 10.4.

## 5. Requisitos funcionales y no funcionales

### 5.1. Requisitos funcionales

| Código | Requisito | Prioridad | Estado |
|--------|-----------|-----------|--------|
| RF-01 | El usuario puede seleccionar un archivo CSV arrastrándolo a la zona de carga o desde un selector de archivos. | Alta | Maqueta |
| RF-02 | La aplicación rechaza archivos que no sean `.csv`, que estén vacíos o que superen el tamaño máximo (propuesta: 5 MB). | Alta | Pendiente |
| RF-03 | El backend calcula el número de filas y columnas del archivo. | Alta | Pendiente |
| RF-04 | El backend detecta el tipo de dato de cada columna (numérico, texto, fecha o booleano). | Alta | Pendiente |
| RF-05 | El backend calcula los valores vacíos por columna (cantidad y porcentaje) y la distribución de vacíos por fila. | Alta | Pendiente |
| RF-06 | La aplicación muestra el informe en tarjetas de resumen y en una tabla de columnas. | Alta | Maqueta |
| RF-07 | El usuario puede pedir sugerencias de análisis y de tratamiento de vacíos a un modelo de OpenRouter. | Media | Pendiente |
| RF-08 | Si el modelo no responde o devuelve un formato no válido, se mantiene el informe y se muestra un aviso. | Alta | Pendiente |

### 5.2. Requisitos no funcionales

| Código | Requisito | Descripción |
|--------|-----------|-------------|
| RNF-01 | Privacidad | Ni el archivo ni sus filas ni sus valores originales se envían a OpenRouter. Solo se envía el resumen estadístico. |
| RNF-02 | Seguridad de la clave | La clave de OpenRouter se guarda en el servidor y nunca llega al navegador ni al repositorio. |
| RNF-03 | Rendimiento | Un CSV de hasta 5 MB debe analizarse en menos de 5 segundos (objetivo propuesto). |
| RNF-04 | Orientación al estudiante | Las sugerencias del modelo se presentan como orientativas y no modifican el archivo. |
| RNF-05 | Usabilidad | La interfaz muestra estados de carga, resultado y error, y funciona en pantallas de móvil sin scroll horizontal. |
| RNF-06 | Mantenibilidad | Cada bloque visual es un componente independiente y los estilos están centralizados en `main.css`. |
| RNF-07 | Pruebas | El backend y los componentes principales tienen pruebas automatizadas. |
| RNF-08 | Coste controlado | Se usa un único modelo configurable y se limita el tamaño de la respuesta solicitada. |

## 6. Diagramas UML del sistema

Los diagramas están escritos en **Mermaid**. Cada uno tiene su archivo fuente `.mmd` en la carpeta [diagramas/](diagramas/) y se ha exportado a SVG para que se vea en cualquier visor. Para cambiar un diagrama, edita su `.mmd` y vuelve a exportarlo. NO ME MOSTRABA NI CON LA PROPIA EXTENSIÓN, PERO SE SUPONE QUE DE NORMAL SERÍA ESTE PASO MUCHO MÁS INTUITIVO Y SENCILLO, PERO VAYA, SOLO QUE ESTÁN EN ARCHIVOS EXTERNOS, NO PROBLEMA.

> Mermaid no tiene un diagrama de casos de uso propio. El diagrama 6.1 se representa con un `flowchart`, siguiendo la notación habitual de actores y casos de uso.

### 6.1. Diagrama de casos de uso

![Diagrama de casos de uso](diagramas/6_1_casos_de_uso.svg)

Código fuente: [diagramas/6_1_casos_de_uso.mmd](diagramas/6_1_casos_de_uso.mmd)


### 6.2. Diagrama de clases

![Diagrama de clases](diagramas/6_2_clases.svg)

Código fuente: [diagramas/6_2_clases.mmd](diagramas/6_2_clases.mmd)


### 6.3. Diagrama de actividad

![Diagrama de actividad](diagramas/6_3_actividad.svg)

Código fuente: [diagramas/6_3_actividad.mmd](diagramas/6_3_actividad.mmd)


### 6.4. Diagrama de secuencia

![Diagrama de secuencia](diagramas/6_4_secuencia.svg)

Código fuente: [diagramas/6_4_secuencia.mmd](diagramas/6_4_secuencia.mmd)



## 7. Plan de ejecución paso a paso

Estado a fecha de esta documentación: la **Fase 1** está completada según el `README.md`. Las fases 2 a 6 están pendientes.

### Fase 1: Inicialización y alcance

- [x] Crear el proyecto Vue 3 + Vite en `frontend/` (JavaScript).
- [x] Maquetar los componentes visuales: `AppHeader`, `CsvDropzone`, `SummaryPanel` y `SuggestionsPanel`.
- [x] Centralizar tipografía, colores y modo oscuro en `main.css`.
- [x] Redactar la introducción, el alcance y la documentación de diseño.

### Fase 2: Análisis del CSV en Python

- [ ] Crear `backend/` con FastAPI, pandas, Pydantic, python-multipart y Uvicorn.
- [ ] Leer el CSV en UTF-8 con respaldo en Latin-1 y detectar el separador.
- [ ] Calcular filas, columnas, tipos, vacíos por columna y distribución de vacíos por fila (RF-03 a RF-05).
- [ ] Exponer `POST /api/analizar` con validación de extensión y tamaño (RF-02).
- [ ] Probar con CSV válidos, vacíos y no válidos.

### Fase 3: Integración con OpenRouter

- [ ] Configurar el cliente de OpenAI SDK con la URL de OpenRouter y la clave en una variable de entorno.
- [ ] Construir el resumen estadístico sin filas ni valores originales (RNF-01).
- [ ] Definir el prompt y validar la respuesta JSON con Pydantic.
- [ ] Exponer `POST /api/sugerencias` con tiempo de espera y manejo de errores (RF-07, RF-08).
- [ ] Probar con respuestas simuladas, sin consumir el modelo real.

### Fase 4: Desarrollo de la interfaz Vue

- [ ] Conectar `CsvDropzone` para leer el archivo y enviarlo al backend (RF-01).
- [ ] Mostrar los datos reales en `SummaryPanel` en lugar de los valores de relleno (RF-06).
- [ ] Mostrar las sugerencias en `SuggestionsPanel` con el aviso de orientativas (RNF-04).
- [ ] Gestionar los estados de carga, resultado y error (RNF-05).
- [ ] Decidir si se migra a TypeScript antes de que el código crezca.

### Fase 5: Pruebas

- [ ] Pruebas del backend con Pytest y HTTPX.
- [ ] Pruebas de componentes con Vitest, Vue Test Utils y jsdom.
- [ ] Prueba específica que compruebe que el prompt no contiene filas ni valores originales (RNF-01).

### Fase 6: Configuración y despliegue

Vercel no puede ejecutar el backend tal y como está planteado (razones detalladas en 10.4), así que el despliegue se hace en dos proveedores distintos durante la fase de pruebas, con la idea de simplificarlo más adelante en una única plataforma.

**Fase 6a — Pruebas con frontend en Vercel y backend en AWS:**

- [ ] Crear `requirements.txt` y un `Dockerfile` para el backend.
- [ ] Desplegar el backend dockerizado en AWS (App Runner o ECS/Fargate) con sus variables de entorno (`OPENROUTER_API_KEY`, etc.).
- [ ] Añadir CORS en FastAPI para aceptar peticiones desde el dominio de Vercel.
- [ ] Configurar las variables de entorno del frontend en Vercel (URL pública del backend en AWS).
- [ ] Desplegar la interfaz (build de Vite) en Vercel.
- [ ] Verificar la aplicación publicada con un CSV de prueba.

**Fase 6b — Todo en AWS (previsto más adelante, también como pruebas):**

- [ ] Servir el build del frontend desde AWS (por ejemplo, S3 + CloudFront, o el propio contenedor con Nginx) en lugar de Vercel.
- [ ] Mantener backend y frontend en el mismo proveedor para simplificar variables de entorno y evitar CORS entre dominios distintos.
- [ ] Comparar coste y complejidad frente al reparto Vercel + AWS de la fase 6a.

## 8. Solución técnica

### 8.1. Configuración del proyecto

- **Backend:** entorno virtual de Python 3.12, con las dependencias listadas en `backend/requirements.txt`.
- **Frontend:** Vue 3 y Vite, instalados con `npm install` dentro de `frontend/`. El `package.json` actual no incluye TypeScript ni Tailwind; si se añaden, se documentará aquí.
- **Comunicación:** en desarrollo, Vite redirige las peticiones `/api` al backend en `http://localhost:8000`, para que el navegador no note dos orígenes distintos.

### 8.2. Backend y análisis de CSV

El backend expone `POST /api/analizar`, que recibe el archivo como campo `archivo` de un formulario multipart.

Pasos del análisis:

1. Comprobar la extensión `.csv` y el tamaño máximo (RF-02).
2. Leer el contenido en UTF-8; si falla, reintentar en Latin-1.
3. Detectar el separador con `csv.Sniffer` y usar coma por defecto.
4. Calcular con pandas:
   - Filas y columnas.
   - Tipo de cada columna: numérico, texto, fecha o booleano.
   - Vacíos por columna: número y porcentaje, contando celdas nulas y cadenas vacías.
   - Distribución de vacíos por fila: cuántas filas tienen 0, 1, 2… celdas vacías.
5. Devolver el `InformeCsv` validado con Pydantic.

Boceto de la función principal (no es código definitivo):

```python
def analizar(contenido: bytes) -> InformeCsv:
    df = leer_csv(contenido)                  # UTF-8 con respaldo Latin-1
    vacios = df.isna().sum()                  # celdas vacías por columna
    por_fila = df.isna().sum(axis=1).value_counts().sort_index()
    return InformeCsv(
        filas=len(df),
        columnas=df.shape[1],
        columnas_detalle=[detalle_columna(df, c) for c in df.columns],
        distribucion_vacios_por_fila={int(k): int(v) for k, v in por_fila.items()},
    )
```

### 8.3. Integración con OpenRouter

- **Cliente:** OpenAI Python SDK con `base_url` apuntando a OpenRouter. La clave y el modelo se leen de variables de entorno (ver 10.3).
- **Qué se envía:** solo el `ResumenEstadistico`, es decir, dimensiones, nombres de columna, tipo, número y porcentaje de vacíos y distribución de vacíos por fila.
- **Qué NO se envía:** filas, valores originales, valores de ejemplo ni categorías concretas. Esta es una decisión de diseño que refuerza RNF-01.
- **Respuesta:** el modelo debe devolver JSON. Se valida con Pydantic (`Sugerencias`). Si el JSON no es válido, el servidor responde `503` y la interfaz conserva el informe (RF-08).
- **Control de coste:** tiempo de espera fijo y límite de tokens de salida (RNF-08).

### 8.4. Interfaz de usuario

| Componente | Responsabilidad | Estado |
|------------|-----------------|--------|
| `AppHeader.vue` | Título y descripción de la aplicación. | Maqueta |
| `CsvDropzone.vue` | Selección o arrastre del CSV y envío al backend. | Maqueta, sin lógica |
| `SummaryPanel.vue` | Tarjetas de filas, columnas, tipos y vacíos, más la tabla por columna. | Maqueta con relleno |
| `SuggestionsPanel.vue` | Botón para pedir sugerencias y lista de preguntas y tratamientos. | Hueco vacío |

Los estados de la interfaz siguen el diagrama 6.5: inicial, cargando, error, informe, sugiriendo, completo e informe sin IA.

### 8.5. Tipos y contrato de datos

Ejemplo de respuesta de `POST /api/analizar`:

```json
{
  "filas": 500,
  "columnas": 6,
  "separador": ",",
  "columnas_detalle": [
    { "nombre": "edad", "tipo": "numerico", "vacios": 20, "porcentaje_vacios": 4.0 }
  ],
  "distribucion_vacios_por_fila": { "0": 463, "1": 30, "2": 7 }
}
```

Ejemplo de respuesta de `POST /api/sugerencias`:

```json
{
  "preguntas_analisis": [
    "¿La edad influye en la nota final?"
  ],
  "tratamientos": [
    { "columna": "edad", "opciones": ["Comprobar por qué faltan los valores", "Imputar con la mediana", "Excluir las filas afectadas"] }
  ],
  "aviso": "Sugerencias orientativas. Decide según el contexto de tu trabajo."
}
```

Los modelos Pydantic (`InformeCsv`, `ColumnaInfo`, `Sugerencias`) y los tipos de TypeScript, si se adoptan, deben mantener la misma forma que estos ejemplos.

## 9. Pruebas automatizadas

### 9.1. Pruebas del backend

Herramientas: Pytest y HTTPX (cliente de prueba de FastAPI).

| Caso | Entrada | Resultado esperado |
|------|---------|--------------------|
| CSV válido | Archivo con cabecera y filas | 200 con filas, columnas y vacíos correctos |
| CSV con vacíos | Celdas vacías en varias columnas | Porcentajes y distribución por fila correctos |
| Archivo no CSV | `.txt` | 400 con mensaje de validación |
| Archivo vacío | 0 bytes | 400 con mensaje de validación |
| Archivo demasiado grande | Más de 5 MB | 413 o 400 con mensaje de validación |
| Sugerencias con IA simulada | Respuesta JSON válida | 200 con `Sugerencias` |
| Sugerencias con IA caída | Error simulado | 503 y sin datos inventados |
| Privacidad del prompt | Informe con valores conocidos | El prompt no contiene ninguno de esos valores |

### 9.2. Pruebas del frontend

Herramientas previstas: Vitest, Vue Test Utils y jsdom. No están instaladas todavía.

- `CsvDropzone`: emite el evento con el archivo seleccionado y muestra error con un archivo no válido.
- `SummaryPanel`: muestra los datos recibidos en lugar de los valores de relleno.
- `SuggestionsPanel`: muestra el aviso de orientativas y el mensaje cuando la IA no está disponible.

## 10. Guía de ejecución y despliegue

### 10.1. Ejecución local

Backend (desde `backend/`):

```bash
python -m venv .venv
source .venv/bin/activate        # En Windows: .venv\Scripts\activate
pip install -r requirements.txt
uvicorn app.main:app --reload --port 8000
```

Frontend (desde `frontend/`):

```bash
npm install
npm run dev
```

Abrir `http://localhost:5173` en el navegador.

### 10.2. Ejecución de pruebas

```bash
# Backend, desde backend/
pytest

# Frontend, desde frontend/ (cuando estén instaladas las pruebas)
npm run test
```

### 10.3. Variables de entorno

| Variable | Dónde | Obligatoria | Descripción |
|----------|-------|-------------|-------------|
| `OPENROUTER_API_KEY` | Backend (`.env` local o variables de entorno del servicio en AWS) | Solo para sugerencias reales | Clave de OpenRouter. Nunca va al repositorio ni al navegador. |
| `OPENROUTER_MODEL` | Backend | No | Identificador del modelo a usar. |
| `OPENROUTER_BASE_URL` | Backend | No | URL de la API de OpenRouter. Por defecto, la oficial. |
| `CSV_TAMANO_MAXIMO_MB` | Backend | No | Tamaño máximo del archivo. Por defecto, 5. |
| `VITE_API_URL` | Frontend (variable de entorno en Vercel) | Sí, en el despliegue dividido | URL pública del backend en AWS, para que el frontend sepa a dónde enviar las peticiones (en local se sustituye por el proxy de Vite a `localhost:8000`). |

Incluir un fichero `.env.example` con los nombres, sin valores, y añadir `.env` al `.gitignore`.

### 10.4. ¿Por qué no se puede ejecutar únicamente en Vercel?

Vercel no es compatible con este backend tal y como está diseñado, por varios motivos:

- **Vercel ignora Docker:** aunque se dockerice el backend para desplegarlo en AWS, ese mismo contenedor no sirve para Vercel. Vercel no construye ni ejecuta un `Dockerfile` de aplicación para servir tráfico; su soporte de Python es a través de su propio runtime de funciones, no de contenedores a medida. Por eso Docker no es "portable" entre Vercel y AWS: son dos modelos de despliegue distintos.


Por eso, para las pruebas, el proyecto se despliega en dos proveedores (Fase 6a): la interfaz en Vercel y el backend dockerizado en AWS. Más adelante (Fase 6b) está previsto mover también el frontend a AWS para tener todo el proyecto en un único proveedor, una vez comprobado que el reparto actual funciona.

### 10.5. Despliegue

**Fase de pruebas — frontend en Vercel, backend en AWS:**

1. Subir el repositorio a GitHub.
2. Backend: crear `requirements.txt` y un `Dockerfile` en `backend/`, y desplegarlo en AWS (App Runner o ECS/Fargate) con sus variables de entorno (10.3).
3. Añadir CORS en FastAPI para aceptar el dominio del frontend en Vercel.
4. Frontend: crear el proyecto en Vercel, elegir `frontend/` como raíz de compilación y configurar `VITE_API_URL` con la URL pública del backend en AWS.
5. Comprobar que `/api/analyze` y `/api/sugerencias` responden desde el dominio de Vercel.
6. Probar la aplicación publicada con un CSV de prueba y con la IA desactivada.

**Fase posterior (prevista) — todo en AWS:**

1. Servir también el build del frontend desde AWS (por ejemplo, un bucket S3 con CloudFront, o el mismo contenedor con Nginx delante de FastAPI).
2. Mantener backend y frontend en el mismo proveedor para simplificar variables de entorno y evitar la configuración de CORS entre dominios distintos.
3. Repetir las comprobaciones del punto anterior sobre este nuevo despliegue.